<a href="https://colab.research.google.com/github/Shrideshi1/multi-label-email-risk-detection/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Multi_Label_Email_Risk_Detection"
DATA_RAW_DIR = f"{PROJECT_DIR}/data/raw"
DATA_PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
MODELS_DIR = f"{PROJECT_DIR}/models"
REPORTS_DIR = f"{PROJECT_DIR}/reports"

os.chdir(PROJECT_DIR)

print("Current folder:", os.getcwd())
print("Processed files:", os.listdir(DATA_PROCESSED_DIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current folder: /content/drive/MyDrive/Multi_Label_Email_Risk_Detection
Processed files: ['combined_dataset.csv', 'features', 'synthetic_confidential.csv']


In [ ]:
combined_df = pd.read_csv(f"{DATA_PROCESSED_DIR}/combined_dataset.csv")

print("Dataset shape:", combined_df.shape)
display(combined_df.head())

In [ ]:
risk_columns = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]

X_text = combined_df["text"].fillna("")
y = combined_df[risk_columns]

print("Text samples:", X_text.shape)
print("Label matrix:", y.shape)
display(y.sum().sort_values(ascending=False).rename("Label Count"))

In [ ]:
print(combined_df.shape)
print(combined_df.columns.tolist())

In [ ]:
print("Missing values:")
print(combined_df[["text"] + risk_columns].isnull().sum())

print("Risk labels per row:")
display(
    y.sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("Number of Risk Labels")
    .reset_index(name="Rows")
)

In [ ]:
print("Dataset loaded successfully.")
print("X_text:", X_text.shape)
print("y:", y.shape)

## Text Preprocessing

To reduce vocabulary sparsity and standardize email content, the preprocessing pipeline applies the following transformations:

- Converts all text to lowercase.
- Replaces URLs with the placeholder token `URL`.
- Replaces email addresses with the placeholder token `EMAIL`.
- Replaces numerical values with the placeholder token `NUMBER`.
- Removes formatting artifacts such as separator lines and unnecessary symbols.
- Normalizes whitespace.

These transformations preserve important structural patterns while reducing variations in the text representation.

In [ ]:
# ------------------------------------------------------------
# Text Cleaning
# ------------------------------------------------------------
# This function standardizes text before feature extraction.
# It lowercases all text, replaces URLs/emails/numbers with
# placeholders, removes unusual symbols, removes email formatting artifacts, long underscore
# separators, code-like underscore tokens, URLs, emails, and numbers.
# normalizes spacing.

import re

def clean_text(text):
    #Lowercase conversion
    text = str(text).lower()

    #Replace URLs and email addresses with placeholders
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)

    #Remove long separator lines made of underscores/dashes/equal signs
    text = re.sub(r"[_\-=\*]{3,}", " ", text)

    #Replace code-style underscore tokens with spaces
    text = re.sub(r"\b_+\w+_*\b", " ", text)
    text = re.sub(r"\b\w+_+\w+\b", " ", text)

    #Replace numbers with a placeholder
    text = re.sub(r"\d+", " NUMBER ", text)

    #Keep only letters, numbers, spaces, periods, dollar signs, and dashes
    text = re.sub(r"[^a-zA-Z0-9\s\.\-$]", " ", text)

    #Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

combined_df["clean_text"] = combined_df["text"].apply(clean_text)

display(combined_df[["text", "clean_text"]].head())

In [ ]:
# ------------------------------------------------------------
# Train/Test Split
# ------------------------------------------------------------
# Split before feature engineering to prevent information leakage.
# The training set is used to learn TF-IDF vocabulary and scaling
# parameters. The test set is only used for evaluation.

from sklearn.model_selection import train_test_split


train_indices, test_indices = train_test_split(
    combined_df.index,
    test_size=0.2,
    random_state=42,
    stratify=combined_df["source"]
)


train_df = combined_df.loc[train_indices].copy()
test_df = combined_df.loc[test_indices].copy()


y_train = y.loc[train_indices]
y_test = y.loc[test_indices]


print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

In [ ]:
# ------------------------------------------------------------
# Metadata Features
# ------------------------------------------------------------
# Metadata features are extracted from original text to preserve
# structural signals such as length, capitalization, and numbers.

def create_metadata_features(df):

    return pd.DataFrame({

        "text_length": df["text"].astype(str).str.len(),

        "word_count": df["text"].astype(str).str.split().str.len(),

        "num_digits": df["text"].astype(str).str.count(r"\d"),

        "num_dollar_signs": df["text"].astype(str).str.count(r"\$"),

        "num_uppercase": df["text"].astype(str).str.count(r"[A-Z]"),

        "num_exclamation": df["text"].astype(str).str.count(r"!"),

        "num_question": df["text"].astype(str).str.count(r"\?")

    })


metadata_train = create_metadata_features(train_df)

metadata_test = create_metadata_features(test_df)


display(metadata_train.head())

In [ ]:
# ------------------------------------------------------------
# Rule-Based Indicator Features
# ------------------------------------------------------------
# These binary features check whether each message contains terms
# associated with specific risk types. They are not the final labels,
# they are additional input features that may help the ML model.

def create_rule_features(df):

    text_lower = df["text"].astype(str).str.lower()

    return pd.DataFrame({

        "has_attachment_ext": text_lower.str.contains(
            r"\.pdf|\.docx|\.xlsx|\.csv|\.zip|\.pptx",
            regex=True
        ).astype(int),


        "has_money": text_lower.str.contains(
            r"\$| revenue | budget | valuation | invoice | payment ",
            regex=True
        ).astype(int),


        "has_credential_terms": text_lower.str.contains(
            r"password|token|api key|secret|credential|oauth|vpn|encryption key|root",
            regex=True
        ).astype(int),


        "has_customer_terms": text_lower.str.contains(
            r"customer|client|account|employee|vendor|partner|payroll",
            regex=True
        ).astype(int),


        "has_legal_terms": text_lower.str.contains(
            r"contract|nda|liability|clause|agreement|compliance|legal|indemnification",
            regex=True
        ).astype(int),


        "has_internal_terms": text_lower.str.contains(
            r"internal|confidential|do not forward|do not distribute|not for circulation",
            regex=True
        ).astype(int)

    })


rule_train_features_df = create_rule_features(train_df)

rule_test_features_df = create_rule_features(test_df)


display(rule_train_features_df.head())

In [ ]:
# ------------------------------------------------------------
# TF-IDF Text Features
# ---------------------------------------------------------------
# TF-IDF converts cleaned text into numerical features based on
# word importance.
#
# ngram_range=(1,2) allows the model to learn:
#   - single words (unigrams)
#       Example: "password"
#
#   - two-word phrases (bigrams)
#       Example: "api key"
#
#
# fit_transform() is applied only to training text.
# The test set uses transform() with the same vocabulary.

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    stop_words="english"
)

# Learn vocabulary and IDF values from training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_df["clean_text"]
)

# Apply learned vocabulary to unseen test data
X_test_tfidf = tfidf_vectorizer.transform(
    test_df["clean_text"]
)

# ------------------------------------------------------------
# Verify TF-IDF Output
# ------------------------------------------------------------

print("Training TF-IDF feature matrix shape:", X_train_tfidf.shape)
print("Testing TF-IDF feature matrix shape:", X_test_tfidf.shape)

print(
    "Number of TF-IDF features:",
    len(tfidf_vectorizer.get_feature_names_out())
)

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Sample TF-IDF features:")
print(feature_names[:50])

In [ ]:
# ------------------------------------------------------------
# Combine TF-IDF, Metadata, and Rule-Based Features
# ------------------------------------------------------------
# TF-IDF is sparse. Metadata and rule-based features are dense,
# so convert them to sparse matrices before combining.

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler


# Scale metadata features
# Fit only on training data

scaler = StandardScaler()


metadata_train_scaled = scaler.fit_transform(
    metadata_train
)


metadata_test_scaled = scaler.transform(
    metadata_test
)


metadata_train_sparse = csr_matrix(
    metadata_train_scaled
)

metadata_test_sparse = csr_matrix(
    metadata_test_scaled
)


# Convert rule features to sparse format

rule_train_sparse = csr_matrix(
    rule_train_features_df.values
)


rule_test_sparse = csr_matrix(
    rule_test_features_df.values
)


# Final training feature matrix

X_train_features = hstack([
    X_train_tfidf,
    metadata_train_sparse,
    rule_train_sparse
])


# Final testing feature matrix

X_test_features = hstack([
    X_test_tfidf,
    metadata_test_sparse,
    rule_test_sparse
])


print("Training feature matrix:", X_train_features.shape)
print("Testing feature matrix:", X_test_features.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

In [ ]:
# ------------------------------------------------------------
# Save Feature Engineering Outputs
# ------------------------------------------------------------

FEATURE_DIR = f"{PROJECT_DIR}/data/processed/features"

os.makedirs(FEATURE_DIR, exist_ok=True)


joblib.dump(
    X_train_features,
    f"{FEATURE_DIR}/X_train_features.pkl"
)


joblib.dump(
    X_test_features,
    f"{FEATURE_DIR}/X_test_features.pkl"
)


joblib.dump(
    y_train,
    f"{FEATURE_DIR}/y_train.pkl"
)


joblib.dump(
    y_test,
    f"{FEATURE_DIR}/y_test.pkl"
)


joblib.dump(
    tfidf_vectorizer,
    f"{FEATURE_DIR}/tfidf_vectorizer.pkl"
)


joblib.dump(
    scaler,
    f"{FEATURE_DIR}/metadata_scaler.pkl"
)


print("Saved feature engineering outputs.")

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

[x for x in feature_names if "url" in x.lower() or "email" in x.lower() or "number" in x.lower()]

We started the feature engineering notebook by loading the final processed multi-label dataset from combined_dataset.csv. The dataset contains 66,910 text samples and 7 target risk labels which are financial risk, credential risk, customer information risk, proprietary risk, legal risk, attachment risk, and phishing/spam risk.
We then created three groups of input features for the machine learning models. First, we cleaned the raw text by lowercasing, removing noisy formatting artifacts, replacing URLs/emails/numbers with standard tokens, and reducing unnecessary symbols. Then, we created metadata features such as text length, word count, digit count, dollar-sign count, uppercase count, and punctuation counts. Third, we added rule-based indicator features that detect risk-related terms such as attachment extensions, financial language, credential terms, customer/account terms, legal terms, and internal/confidential routing phrases.
Finally, we generated TF-IDF text features using word and phrase patterns from the cleaned text. We improved the text cleaning after noticing noisy underscore/code-like tokens in the initial TF-IDF output. The final feature set combines TF-IDF features, metadata features, and rule-based indicators, and is ready to be saved for the model training notebook.